# Riemanian Flow Matching on S2 Surface

## Goal and Setup

We are using a simple geometry called $S^2$ to test the Riemanian Flow Matching algorithm. RFM is trying to learn targets on non-euclidean geometries eg., earthquakes on the earth's surface.

The surface of a sphere is $S^2$. The geometry rules of S2 differ from classsical Euclidean. A point on $S^2$ lies on the surface of a unit sphere. An $S^2$ surface is a 2D geometry embedded in a 3D space. 

x0 and x1 are the start and end points. x1 is ussually sampled from a target data distribution which does not have an analytical definition.

We are using this simple geometry to build intuition into how this method works.





## Imports 

In [1567]:
import torch

## Sample Points ON Standard radius r=1 Sphere

We randomly sample z vertical dimension. Based on the sampled z, we find that at each latitude we have circles of differing size. We calculate the radius of that circle, and then within that circle, we choose an angle between 0 and 2π. Based on the angle, we get the x and y dimensions.

In [1568]:
def sample_sphere(n):
    """Sample a point on a sphere"""
    z = 2*torch.rand(n) - 1
    phi = 2*(torch.pi)*torch.rand(n)
    r = torch.sqrt(1-z**2)
    x, y = r*torch.cos(phi), r*torch.sin(phi)
    return torch.concat([x.reshape(-1,1),y.reshape(-1,1),z.reshape(-1,1)], axis=1)
    

In [1569]:
coord = sample_sphere(4)

In [1570]:
coord

tensor([[ 0.2229, -0.8120,  0.5394],
        [ 0.0431,  0.8793, -0.4743],
        [-0.6011,  0.1475, -0.7854],
        [ 0.4642,  0.8629,  0.1997]])

In [1571]:
torch.square(coord)

tensor([[0.0497, 0.6593, 0.2910],
        [0.0019, 0.7732, 0.2250],
        [0.3614, 0.0217, 0.6169],
        [0.2155, 0.7447, 0.0399]])

Here we are checking the validity of all the coordinates by seeing if the sum of squares add up to one, basically if all the coordinates are lying on the surface of the unit sphere.

In [1572]:
def check_validity(coord):
    assert  torch.all(torch.abs(torch.sum((torch.square(coord)), axis=1)  - 1) <= 1e-3)

In [1573]:
check_validity(coord)

In [1574]:
x,y = sample_sphere(4), sample_sphere(4)

In [1575]:
x.shape, y.shape

(torch.Size([4, 3]), torch.Size([4, 3]))

## Log Map
We are basically finding the minimum geodesic or minor arc for going from one point to another on the circle. How we do this is the following:
1. We find the dot product of x and y.
2. Based on the dot product, we find the inverse cosine.
3. We find the parallel component of y along x.  We're doing this because we want to find, ultimately, the perpendicular component of y along x.
4. We find the perpendicular component along y by subtracting y and y parallel. 
5. Then we normalize the perpendicular component.
6. Then we get the vector v as the normalized y-perpendicular component times theta.

In [1576]:

def log_map(x,y):
    """Finds the minimum geodesic or minor arc for circle"""
    assert x.shape==y.shape
    n = x.shape[0]
    dot_prod = torch.tensor([torch.dot(x[i],y[i]) for i in range(n)]).reshape(-1,1)
    theta = torch.tensor([torch.arccos(dot) for dot in dot_prod]).reshape(-1,1)
    y_parallel = dot_prod * x
    y_perp = y - y_parallel
    y_perp_norm = torch.nn.functional.normalize(y_perp, 2, dim=-1)
    v = theta * y_perp_norm
    return v


        

In [1577]:
v = log_map(x,y)

In [1578]:
v.shape

torch.Size([4, 3])

A small wellness check to test whether X and V are tangential or not

In [1579]:
def test_tangentness(x, v, tol=1e-5):
    assert x.shape==v.shape
    n = x.shape[0]
    dot_prod = torch.tensor([torch.dot(x[i],v[i]) for i in range(n)])
    tol_check = torch.abs(dot_prod)<=tol
    assert torch.all(tol_check)


In [1580]:
test_tangentness(x,v)

Just when I thought I got it, I fall in love until you say we're testing whether the L2 norm of V and theta are the same or not.

In [1581]:
def log_map_length_test(x,y,v, tol=1e-5):
    assert x.shape == y.shape
    assert x.shape == v.shape
    n = x.shape[0]
    theta = torch.tensor([torch.arccos(torch.dot(x[i], y[i])) for i in range(n)])
    v_norm = torch.linalg.vector_norm(v, ord=2, dim=-1)
    assert torch.all(torch.abs(v_norm - theta) <= tol)


In [1582]:
log_map_length_test(x,y,v)

## Exp Map
The exponential map starts at a point `x` on the sphere and follows a tangent vector `v` to produce a new point `y` on the sphere.

The length of `v`, `theta = ||v||`, is the angular distance we travel along the great circle. The unit tangent direction `v / ||v||` and the radius vector `x` are perpendicular, so they span the two-dimensional plane containing that great circle.

Inside this plane, the new point has a cosine component along `x` and a sine component along the unit tangent direction. Therefore, the construction is `y = cos(theta) x + sin(theta) v/||v||`.

The tests below check that this construction inverts `log_map` and keeps the result on the unit sphere.

In [1583]:
def exp_map(x,v):
    assert x.shape==v.shape
    n = x.shape[0]
    v_norm = torch.nn.functional.normalize(v, p=2, dim=-1)
    theta = torch.linalg.vector_norm(v, ord=2, dim=-1)
    y = torch.cos(theta).reshape(-1,1) * x + torch.sin(theta).reshape(-1,1) * v_norm
    return y
    

In [1584]:
exp_map(x,v)

tensor([[-0.7885, -0.5624, -0.2488],
        [ 0.3759,  0.1876,  0.9075],
        [-0.1350,  0.9796,  0.1485],
        [-0.9337, -0.3516,  0.0677]])

Here we are testing whether log map and exp map are truly inverse. 

In [1585]:
def test_exp(x,y, tol=1e-5):
    assert torch.all(torch.abs(exp_map(x, log_map(x,y)) - y) <= tol)

In [1586]:
test_exp(x,y)

We are testing whether the new point still lies on the sphere. 

In [1587]:
def test_unit_norm(x,v, tol=1e-5):
    assert torch.all(torch.abs(torch.linalg.vector_norm(exp_map(x,v), ord=2, dim=-1)-1) <= tol)

In [1588]:
test_unit_norm(x,v)

We are creating a distance metric for the Riemannian function. The best distance function for this is just the inverse cosine of the dot product or the angle theta. 

In [1589]:
def premetric_d(x,y):
    assert x.shape == y.shape
    n = x.shape[0]
    return torch.tensor(
        [torch.arccos(torch.clamp(torch.dot(x[i], y[i]), min=-1, max=1)) for i in range(n)]
        )

In [1590]:
premetric_d(x,y)

tensor([2.7783, 0.6344, 1.4752, 1.4195])

Testing a property where the distance between the same point is zero 

In [1591]:
def test_d_x_x_zero(x,tol=1e-3):
    assert torch.all(
        torch.abs(
            premetric_d(x,x) - 0
        ) <= tol
    )

In [1592]:
test_d_x_x_zero(x)

Testing the symmetry property where the distance from x to y and y to x are the same. 

In [1593]:
def test_d_symmetry(x,y,tol=1e-3):
    assert torch.all(
        torch.abs(
            premetric_d(x,y) - premetric_d(y,x)
        ) <= tol
    )

In [1594]:
test_d_symmetry(x,y)

Testing that all distances are always non-negative 

In [1595]:
def test_d_non_neg(x,y):
    assert torch.all(
        premetric_d(x,y) >= 0
    )

In [1596]:
test_d_non_neg(x,y)

## Gradient
We are getting the gradient by dividing the vector v by the L2 norm. Here we can put that long derivation we discussed, and it must be in the RFM.md.  

In [1597]:
def grad_d(x, y):
    v = log_map(x,y)
    v_norm = torch.linalg.vector_norm(v, ord=2, dim=-1).reshape(-1,1)
    return -v * torch.reciprocal(v_norm)

In [1598]:
g = grad_d(x,y)

In [1599]:
test_tangentness(x,g)

In [1600]:
test_unit_norm(x,g)

In [1601]:
def test_against_log_map(x,y, tol=1e-3):
    assert torch.all(torch.abs(
        torch.nn.functional.normalize(log_map(x,y),p=2,dim=-1) + grad_d(x,y)
    ) <= tol
    )

In [1602]:
test_against_log_map(x,y)

In [1603]:
def get_time_sched_and_derivative(t):
    assert torch.all(t < 1)
    assert torch.all(t>=0)
    return 1-t, -1*torch.reciprocal(1-t)

In [1604]:
get_time_sched_and_derivative(torch.tensor([0,0.5]))

(tensor([1.0000, 0.5000]), tensor([-1., -2.]))

In [1605]:
def conditional_vf(x, x1, t):
    assert x.shape == x1.shape
    assert x.shape[0] == t.shape[0]
    grad = grad_d(x,x1)
    pre = premetric_d(x,x1).reshape(-1,1)
    _, log_deriv = get_time_sched_and_derivative(t)
    log_deriv = log_deriv.reshape(-1,1)
    return log_deriv * pre * grad * \
            torch.reciprocal(
                torch.square(
                torch.linalg.vector_norm(grad,ord=2,dim=-1)
                ).reshape(-1,1)
                )

    


In [1606]:
def get_time_samples(n):
    return torch.rand(n)

In [1607]:
t = get_time_samples(4)

In [1608]:
u = conditional_vf(x,y,t)

In [1609]:
test_tangentness(x,u)

In [1610]:
t.shape

torch.Size([4])

In [1611]:
def geodesic_path(x0, x1, t):
    return exp_map(x0, t.reshape(-1,1)*log_map(x0,x1))

In [1612]:
geodesic_path(x,y,t)

tensor([[ 0.3194,  0.8341,  0.4497],
        [-0.1349,  0.3772,  0.9163],
        [ 0.3764,  0.9264, -0.0084],
        [-0.5538,  0.2285, -0.8007]])

In [1613]:
def test_start(x, y, tol=1e-3):
    assert x.shape == y.shape
    n = x.shape[0]
    t = torch.zeros(n,1)
    assert torch.all(torch.abs(geodesic_path(x,y,t) - x)<=tol)


In [1614]:
test_start(x,y)

In [1615]:
def test_end(x, y, tol=1e-3):
    assert x.shape == y.shape
    n = x.shape[0]
    t = torch.ones(n,1)
    assert torch.all(torch.abs(geodesic_path(x,y,t) - y)<=tol)

In [1616]:
test_end(x,y)

In [1617]:
def test_distance_schedule(x0, x1, t, tol=1e-3):
    t = t.reshape(-1, 1)
    xt = exp_map(x0, t * log_map(x0, x1))
    assert torch.all(
        torch.abs(
            premetric_d(xt, x1).reshape(-1,1) - ((1-t) * premetric_d(x0, x1).reshape(-1,1))
        ) <= tol
        
    )

In [1618]:
test_distance_schedule(x,y,t)

In [1619]:
t_sweep = torch.tensor([0, 0.25, 0.5, 0.75, 1])
n=4
t_sweep.repeat(n,1)

tensor([[0.0000, 0.2500, 0.5000, 0.7500, 1.0000],
        [0.0000, 0.2500, 0.5000, 0.7500, 1.0000],
        [0.0000, 0.2500, 0.5000, 0.7500, 1.0000],
        [0.0000, 0.2500, 0.5000, 0.7500, 1.0000]])

In [1620]:
def test_sweep(x0, x1):
    assert x0.shape == x1.shape
    n = x0.shape[0]
    t_sweep = torch.tensor([0, 0.25, 0.5, 0.75, 1]).repeat(n, 1)
    n_t = t_sweep.shape[1]
    for i in range(n_t):
        test_distance_schedule(x0, x1, t_sweep[:, i])
       



In [1621]:
test_sweep(x,y)

In [1622]:
def get_batch_geodesics(x0, x1, t_sweep):
    n = x0.shape[0]
    dim = x0.shape[1]
    n_t = t_sweep.shape[0]
    t_mat = t_sweep.repeat(n,1)
    # print(t_mat.shape)
    xt_mat = torch.zeros(n_t, n, dim)
    for i in range(n_t):
        t = t_mat[:, i]
        xt = geodesic_path(x0, x1, t)
        xt_mat[i, :, :] = xt
        # print(xt)
    return xt_mat


In [1623]:
def get_batch_cvf(xt_mat, x1, t_sweep):
    n = x1.shape[0]
    dim = x1.shape[1]
    n_t = t_sweep.shape[0]
    t_mat = t_sweep.repeat(n,1)
    cvf_mat = torch.zeros(n_t, n, dim)
    for i in range(n_t):
        t = t_mat[:, i]
        xt = xt_mat[i, :, :]
        ut = conditional_vf(xt, x1, t)
        cvf_mat[i, :, :] = ut
    return cvf_mat



In [1624]:
def test_conditional_vf_matches_path_velocity(x0, x1, eps=1e-3, tol=1e-3):
    assert x0.shape==x1.shape
    t_sweep = torch.tensor([0.25, 0.5, 0.75])
    t_sweep_plus = t_sweep + eps
    t_sweep_minus = t_sweep - eps

    xt_mat = get_batch_geodesics(x0, x1, t_sweep)
    xt_mat_plus = get_batch_geodesics(x0, x1, t_sweep_plus)
    xt_mat_minus = get_batch_geodesics(x0, x1, t_sweep_minus)

    finite_velocity = (xt_mat_plus - xt_mat_minus)/ (2 * eps)

    cvf_mat = get_batch_cvf(xt_mat, x1, t_sweep)
    
    n_t = t_sweep.shape[0]
    n = x0.shape[0]
    diff = finite_velocity - cvf_mat
    for i in range(n_t):
        for j in range(n):
            assert torch.linalg.vector_norm(diff[i,j,:], ord=2, dim=-1)<=tol
    







    

In [1625]:
test_conditional_vf_matches_path_velocity(x,y)

In [1626]:
def get_batch_euclidean_xt(x0, x1, t_sweep):
    assert x0.shape == x1.shape
    n = x0.shape[0]
    n_t = t_sweep.shape[0]
    dim = x0.shape[1]
    xt_mat = torch.zeros(n_t, n, dim)
    for i, t in enumerate(t_sweep):
        xt = (1-t) * x0 + t * x1
        xt_mat[i, :, :] = xt
    return xt_mat



In [1627]:
def euclidean_test(n, eps=1e-3, tol=1e-3):
    x0 = torch.randn(n, 3)
    x1 = torch.randn(n, 3)
    t_sweep = torch.tensor([0.25, 0.5, 0.75])
    n_t = t_sweep.shape[0]
    n = x0.shape[0]
    dim = x0.shape[1]
    xt_mat = get_batch_euclidean_xt(x0, x1, t_sweep)
    x1_mat = x1.reshape(1, n, dim)
    residual = xt_mat - x1_mat
    print(residual.shape)
    d = torch.linalg.vector_norm(residual, ord=2, dim=2).reshape(n_t, n, 1)
    res_norm = residual / d
    dlogk = -1/(1-t_sweep).reshape(n_t,1,1)
    cvf = dlogk * d * res_norm
    finite_velocity = x1 - x0
    fin_reshape = finite_velocity.reshape(1, n, dim)

    v_diff = fin_reshape - cvf

    for i in range(n_t):
        for j in range(n):
            assert torch.linalg.vector_norm(v_diff[i, j, :], ord=2, dim=-1) <=tol




In [1628]:
euclidean_test(4)

torch.Size([3, 4, 3])
